# Bài tập 3: Spark SQL và Joins

Bài tập này dành cho thực tập sinh làm quen với Apache Spark SQL và các phép Join.

Nội dung bao gồm 3 phần:

- **Task 3.1: SQL Setup and Basic Queries** - Tải dữ liệu CSV, tạo temporary views và viết câu lệnh SQL cơ bản để tính tổng doanh số theo hạng thành viên (loyalty tier).
- **Task 3.2: Complex Analytics Queries** - Viết các câu truy vấn phân tích phức tạp hơn: giá trị vòng đời khách hàng (CLV) theo nhóm tuổi, hiệu suất sản phẩm kèm biên lợi nhuận, và phân tích theo mùa sử dụng Window Functions.
- **Task 3.3: Join Practice** - Thực hành các loại Join: Inner Join, Left Join, Broadcast Join và Self Join.

## Task 3.1: SQL Setup and Basic Queries

Trong phần này, chúng ta sẽ học cách thiết lập môi trường Spark SQL. Đầu tiên, cần khởi tạo một `SparkSession` - đây là điểm vào (entry point) duy nhất để làm việc với mọi chức năng của Spark. Sau đó, chúng ta sẽ tải 3 file CSV (sales_data, customers, products) vào các DataFrame, đăng ký chúng dưới dạng **temporary views** để có thể truy vấn bằng cú pháp SQL thuần túy.

Mục tiêu cuối cùng của Task 3.1 là viết một câu SQL kết hợp (JOIN) bảng bán hàng với bảng khách hàng, sau đó nhóm (GROUP BY) theo hạng thành viên (loyalty_tier) để tính các chỉ số thống kê cơ bản như COUNT, SUM, AVG.

In [ ]:
from pyspark.sql import SparkSession

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("Exercise3") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("SparkSession đã được tạo thành công!")
print("App name:", spark.sparkContext.appName)

**Giải thích cú pháp SparkSession.builder:**

- `SparkSession.builder`: Trả về một đối tượng Builder dùng để cấu hình và xây dựng SparkSession.
- `.appName("Exercise3")`: Đặt tên cho ứng dụng Spark. Tên này sẽ hiển thị trên giao diện Spark UI, giúp dễ dàng nhận diện job.
- `.master("spark://spark-master:7077")`: Chỉ định địa chỉ của Spark Master. Ở đây chúng ta kết nối tới một cụm Spark standalone đang chạy tại `spark-master` cổng `7077`. Nếu muốn chạy local, ta sẽ dùng `local[*]`.
- `.getOrCreate()`: Kiểm tra xem đã có SparkSession nào tồn tại chưa. Nếu có thì trả về session cũ, nếu chưa có thì tạo mới. Điều này tránh việc tạo nhiều session gây xung đột tài nguyên.

**Đầu ra mong đợi:** In ra dòng thông báo "SparkSession đã được tạo thành công!" và tên ứng dụng là "Exercise3".

In [ ]:
# Tải dữ liệu từ các file CSV vào DataFrame
df_sales = spark.read.csv(
    "/home/jovyan/data/csv/sales_data.csv",
    header=True,
    inferSchema=True
)

df_customers = spark.read.csv(
    "/home/jovyan/data/csv/customers.csv",
    header=True,
    inferSchema=True
)

df_products = spark.read.csv(
    "/home/jovyan/data/csv/products.csv",
    header=True,
    inferSchema=True
)

print("Số lượng cột trong df_sales:", len(df_sales.columns))
print("Số lượng cột trong df_customers:", len(df_customers.columns))
print("Số lượng cột trong df_products:", len(df_products.columns))

**Giải thích tham số read.csv:**

- `spark.read.csv(...)`: Phương thức đọc file CSV và trả về một DataFrame.
- Tham số thứ nhất (`"/home/jovyan/data/csv/..."`): Đường dẫn tuyệt đối tới file CSV cần đọc.
- `header=True`: Báo cho Spark biết dòng đầu tiên của file CSV chứa tên các cột. Nếu không có tham số này, Spark sẽ tự đặt tên cột là `_c0`, `_c1`, v.v.
- `inferSchema=True`: Yêu cầu Spark quét qua dữ liệu để tự động suy luận kiểu dữ liệu (Integer, Double, String, Timestamp...) cho từng cột thay vì mặc định đọc tất cả dưới dạng String.

**Đầu ra mong đợi:** In ra số lượng cột của từng DataFrame. Ví dụ: "Số lượng cột trong df_sales: 6", "Số lượng cột trong df_customers: 4", v.v. Các con số cụ thể phụ thuộc vào cấu trúc file CSV thực tế.

In [ ]:
# Đăng ký các DataFrame dưới dạng Temporary Views để dùng SQL
df_sales.createOrReplaceTempView("sales")
df_customers.createOrReplaceTempView("customers")
df_products.createOrReplaceTempView("products")

print("Đã tạo xong các temporary views: sales, customers, products")

**Giải thích Temporary Views:**

- `createOrReplaceTempView("tên_view")`: Phương thức này đăng ký DataFrame dưới dạng một bảng tạm thời trong catalog của SparkSession hiện tại.
- **Temporary** (tạm thời): View này chỉ tồn tại trong suốt vòng đời của SparkSession. Khi tắt SparkSession, view sẽ biến mất. Nó không được lưu trữ vật lý như một bảng Hive hay JDBC.
- **CreateOrReplace**: Nếu view chưa tồn tại thì tạo mới, nếu đã tồn tại thì ghi đè lên. Điều này rất tiện lợi khi bạn chạy lại cell nhiều lần trong Jupyter Notebook mà không bị báo lỗi trùng tên.
- Sau khi tạo view, ta có thể dùng `spark.sql("SELECT * FROM tên_view")` để truy vấn y hệt như đang dùng SQL trên một cơ sở dữ liệu quan hệ.

**Đầu ra mong đợi:** In ra dòng chữ xác nhận "Đã tạo xong các temporary views: sales, customers, products".

In [ ]:
# Task 3.1: SQL query tìm tổng doanh số theo hạng thành viên (loyalty tier)
query_tier_sales = """
    SELECT 
        c.loyalty_tier,
        COUNT(*) AS total_transactions,
        SUM(s.quantity * s.unit_price) AS total_revenue,
        AVG(s.quantity * s.unit_price) AS avg_transaction_value
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    GROUP BY c.loyalty_tier
    ORDER BY total_revenue DESC
"""

df_tier_sales = spark.sql(query_tier_sales)
df_tier_sales.show(truncate=False)

**Phân tích cú pháp SQL:**

- `spark.sql(query_tier_sales)`: Truyền chuỗi SQL vào để Spark Catalyst Optimizer phân tích, lập kế hoạch và thực thi.
- `FROM sales s JOIN customers c ON s.customer_id = c.customer_id`: Phép INNER JOIN giữa bảng `sales` (alias `s`) và `customers` (alias `c`). Chỉ những bản ghi có `customer_id` khớp nhau ở cả 2 bảng mới được giữ lại.
- `SUM(s.quantity * s.unit_price) AS total_revenue`: Tính tổng doanh thu. Vì mỗi dòng bán hàng có số lượng (`quantity`) và đơn giá (`unit_price`), ta nhân chúng lại để ra doanh thu từng giao dịch rồi cộng dồn. `AS` dùng để đặt tên bí danh cho cột kết quả.
- `COUNT(*) AS total_transactions`: Đếm tổng số lượng giao dịch (số dòng) trong mỗi nhóm.
- `AVG(...) AS avg_transaction_value`: Tính giá trị trung bình của mỗi giao dịch trong nhóm.
- `GROUP BY c.loyalty_tier`: Gom các dòng có cùng `loyalty_tier` (hạng thành viên, ví dụ: Gold, Silver, Bronze) vào một nhóm để áp dụng các hàm aggregate (COUNT, SUM, AVG) ở trên.
- `ORDER BY total_revenue DESC`: Sắp xếp kết quả theo tổng doanh thu giảm dần (hạng mang lại nhiều tiền nhất lên đầu).

**Đầu ra mong đợi:** Một bảng có 4 cột: `loyalty_tier`, `total_transactions`, `total_revenue`, `avg_transaction_value`. Mỗi hàng đại diện cho một hạng thành viên, được sắp xếp từ doanh thu cao nhất xuống thấp nhất.

## Task 3.2: Complex Analytics Queries

Phần này nâng cao độ khó với 3 câu truy vấn phân tích:
1. **Customer Lifetime Value (CLV) Analysis**: Phân tích giá trị vòng đời khách hàng chia theo nhóm tuổi và hạng thành viên, sử dụng biểu thức điều kiện `CASE WHEN`.
2. **Product Performance with Profit Margins**: Đánh giá hiệu suất sản phẩm kèm tính toán biên lợi nhuận, yêu cầu lọc kết quả bằng `HAVING`.
3. **Seasonal Analysis with Window Functions**: Phân tích xu hướng theo mùa sử dụng hàm cửa sổ (Window Function) `LAG()` để so sánh doanh thu tháng trước với tháng sau.

In [ ]:
# Query 1: Customer Lifetime Value Analysis by age group
query_clv = """
    SELECT 
        CASE 
            WHEN c.age < 30 THEN 'Young (18-29)'
            WHEN c.age >= 30 AND c.age < 50 THEN 'Middle (30-49)'
            ELSE 'Senior (50+)'
        END AS age_group,
        c.loyalty_tier,
        COUNT(DISTINCT s.customer_id) AS unique_customers,
        SUM(s.quantity * s.unit_price) AS total_revenue,
        AVG(s.quantity * s.unit_price) AS avg_order_value
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    GROUP BY 
        age_group,
        c.loyalty_tier
    ORDER BY 
        age_group,
        total_revenue DESC
"""

df_clv = spark.sql(query_clv)
df_clv.show(truncate=False)

**Phân tích cú pháp Query 1 (CLV):**

- `CASE WHEN ... THEN ... ELSE ... END`: Đây là biểu thức điều kiện trong SQL, tương đương với if-else trong Python. Nó kiểm tra giá trị của cột `c.age`:
  - Nếu `age < 30`: gán nhãn `'Young (18-29)'`
  - Nếu `age >= 30 AND age < 50`: gán nhãn `'Middle (30-49)'`
  - `ELSE` (các trường hợp còn lại, tức là >= 50): gán nhãn `'Senior (50+)'`
  Kết quả được đặt tên là `age_group`.
- `COUNT(DISTINCT s.customer_id)`: Đếm số lượng khách hàng **không trùng lặp** trong mỗi nhóm. Khác với `COUNT(*)` đếm tổng số giao dịch, hàm này đảm bảo một khách hàng mua 10 lần cũng chỉ được tính là 1.
- `GROUP BY age_group, c.loyalty_tier`: Nhóm dữ liệu theo 2 cấp độ. Trước tiên theo nhóm tuổi, sau đó theo hạng thành viên bên trong nhóm tuổi đó.

**Đầu ra mong đợi:** Bảng gồm 5 cột: `age_group`, `loyalty_tier`, `unique_customers`, `total_revenue`, `avg_order_value`. Dữ liệu được sắp xếp theo tên nhóm tuổi (thứ tự chữ cái) và trong mỗi nhóm tuổi thì sắp theo doanh thu giảm dần.

In [ ]:
# Query 2: Product Performance with profit margins
query_product_perf = """
    SELECT 
        s.category,
        s.product_name,
        COUNT(*) AS total_orders,
        SUM(s.quantity * s.unit_price) AS total_revenue,
        SUM(s.quantity * p.cost_price) AS total_cost,
        (SUM(s.quantity * s.unit_price) - SUM(s.quantity * p.cost_price)) AS total_profit
    FROM sales s
    JOIN products p 
        ON s.category = p.category 
        AND s.product_name = p.product_name
    GROUP BY 
        s.category,
        s.product_name
    HAVING COUNT(*) >= 10
    ORDER BY total_profit DESC
"""

df_product_perf = spark.sql(query_product_perf)
df_product_perf.show(truncate=False)

**Phân tích cú pháp Query 2 (Product Performance):**

- `JOIN products p ON s.category = p.category AND s.product_name = p.product_name`: Phép JOIN ở đây sử dụng **2 điều kiện** kết hợp bằng `AND`. Ta phải khớp cả `category` lẫn `product_name` giữa bảng `sales` và `products` để lấy đúng thông tin giá vốn (`cost_price`) của sản phẩm đó.
- `SUM(s.quantity * p.cost_price) AS total_cost`: Tính tổng chi phí (giá vốn). Lấy số lượng bán được nhân với giá vốn của sản phẩm.
- `(SUM(...) - SUM(...)) AS total_profit`: Lợi nhuận = Tổng doanh thu - Tổng chi phí.
- `HAVING COUNT(*) >= 10`: Điểm khác biệt quan trọng so với `WHERE`. `WHERE` lọc dòng **trước khi** nhóm (GROUP BY), còn `HAVING` lọc nhóm **sau khi** đã nhóm và tính toán aggregate. Ở đây ta chỉ giữ lại những sản phẩm có từ 10 đơn hàng trở lên để đảm bảo dữ liệu có ý nghĩa thống kê.

**Đầu ra mong đợi:** Bảng gồm 6 cột: `category`, `product_name`, `total_orders`, `total_revenue`, `total_cost`, `total_profit`. Chỉ chứa các sản phẩm có >= 10 đơn hàng, sắp xếp theo lợi nhuận giảm dần.

In [ ]:
# Query 3: Seasonal Analysis with Window Functions
query_seasonal = """
    WITH monthly_sales AS (
        SELECT 
            category,
            EXTRACT(YEAR FROM sale_date) AS sale_year,
            EXTRACT(MONTH FROM sale_date) AS sale_month,
            SUM(quantity * unit_price) AS monthly_revenue
        FROM sales
        GROUP BY 
            category,
            EXTRACT(YEAR FROM sale_date),
            EXTRACT(MONTH FROM sale_date)
    )
    SELECT 
        category,
        sale_year,
        sale_month,
        monthly_revenue,
        LAG(monthly_revenue, 1) OVER (
            PARTITION BY category 
            ORDER BY sale_year, sale_month
        ) AS prev_month_revenue,
        (monthly_revenue - LAG(monthly_revenue, 1) OVER (
            PARTITION BY category 
            ORDER BY sale_year, sale_month
        )) AS month_over_month_change
    FROM monthly_sales
    ORDER BY category, sale_year, sale_month
"""

df_seasonal = spark.sql(query_seasonal)
df_seasonal.show(truncate=False)

**Phân tích cú pháp Query 3 (Seasonal & Window Functions):**

- `WITH monthly_sales AS (...)`: Đây là CTE (Common Table Expression), giống như tạo một bảng tạm chỉ dùng trong phạm vi câu truy vấn này. Nó giúp code dễ đọc hơn thay vì viết subquery lồng nhau.
- `EXTRACT(YEAR FROM sale_date)` và `EXTRACT(MONTH FROM sale_date)`: Trích xuất năm và tháng từ cột ngày tháng `sale_date` để phục vụ việc nhóm theo tháng/năm.
- `LAG(monthly_revenue, 1)`: Hàm Window Function lấy giá trị của cột `monthly_revenue` ở **1 dòng trước đó** (offset = 1).
- `OVER (PARTITION BY category ORDER BY sale_year, sale_month)`: Định nghĩa "cửa sổ" (window) cho hàm LAG:
  - `PARTITION BY category`: Chia dữ liệu thành các phân vùng độc lập theo từng `category`. Hàm LAG sẽ chỉ nhìn ngược lại trong cùng một category, không bị nhảy sang category khác.
  - `ORDER BY sale_year, sale_month`: Sắp xếp các dòng trong mỗi phân vùng theo thời gian để LAG biết đâu là "dòng trước đó".
- `month_over_month_change`: Lấy doanh thu tháng hiện tại trừ đi doanh thu tháng trước (do LAG trả về) để ra mức độ tăng/giảm.

**Đầu ra mong đợi:** Bảng gồm 6 cột. Cột `prev_month_revenue` sẽ có giá trị `null` ở dòng đầu tiên của mỗi category (vì không có tháng trước đó). Cột `month_over_month_change` cũng sẽ `null` ở dòng đầu, các dòng sau sẽ là số dương (tăng trưởng) hoặc âm (sụt giảm).

## Task 3.3: Join Practice

Phần này thực hành trực tiếp các loại Join trong PySpark DataFrame API (thay vì chỉ dùng SQL):
1. **Inner Join**: Giữ lại các dòng có khóa chung ở cả 2 bảng.
2. **Left Join**: Giữ lại toàn bộ bảng trái, bảng phải không khớp thì điền null.
3. **Broadcast Join**: Tối ưu hóa khi join một bảng lớn với một bảng nhỏ bằng cách gửi bảng nhỏ tới tất cả các worker node.
4. **Self Join**: Join một bảng với chính nó để tìm các mối quan hệ nội bộ (ví dụ: khách hàng mua cùng một sản phẩm).

In [ ]:
# Inner join sales + customers on customer_id
df_inner_join = df_sales.join(
    df_customers,
    on="customer_id",
    how="inner"
)

print("Số lượng dòng sau khi Inner Join:", df_inner_join.count())

**Giải thích Inner Join:**

- `df_sales.join(df_customers, ...)`: Gọi phương thức `join` trên DataFrame `df_sales`, truyền vào DataFrame cần join là `df_customers`.
- `on="customer_id"`: Chỉ định cột dùng làm khóa để ghép 2 bảng. Vì cả 2 DataFrame đều có cột tên `customer_id`, ta có thể truyền thẳng tên chuỗi. Spark sẽ tự hiểu là `df_sales.customer_id == df_customers.customer_id`.
- `how="inner"`: Kiểu join. Inner join chỉ trả về những dòng mà giá trị `customer_id` xuất hiện ở **cả hai** bảng. Nếu một giao dịch trong `sales` có `customer_id` không tồn tại trong `customers`, giao dịch đó sẽ bị loại bỏ.
- `.count()`: Hành động (action) đếm tổng số dòng của DataFrame kết quả.

**Đầu ra mong đợi:** In ra một số nguyên đại diện cho số lượng giao dịch có thông tin khách hàng hợp lệ. Số này thường nhỏ hơn hoặc bằng tổng số dòng của bảng `sales`.

In [ ]:
# Left join sales + customers on customer_id
df_left_join = df_sales.join(
    df_customers,
    on="customer_id",
    how="left"
)

print("Số lượng dòng sau khi Left Join:", df_left_join.count())

**Giải thích Left Join:**

- `how="left"` (hoặc `"left_outer"`): Kiểu Left Join. Khác với Inner Join, Left Join giữ lại **toàn bộ** các dòng từ bảng nằm bên trái (ở đây là `df_sales`).
- Nếu một `customer_id` trong `df_sales` không tìm thấy trong `df_customers`, dòng đó vẫn được giữ lại, nhưng các cột thuộc về `df_customers` (như tên, tuổi, loyalty_tier) sẽ được điền giá trị `null`.
- Do đó, số lượng dòng của Left Join luôn **lớn hơn hoặc bằng** Inner Join, và tối đa bằng số dòng của bảng `sales`.

**Đầu ra mong đợi:** In ra số lượng dòng. Con số này sẽ bằng với tổng số dòng ban đầu của `df_sales` (nếu `customer_id` trong `sales` là duy nhất cho mỗi dòng) hoặc lớn hơn nếu có quan hệ 1-nhiều. So sánh số này với số của Inner Join ở cell trước để thấy sự khác biệt.

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast join sales + products on [category, product_name]
df_broadcast_join = df_sales.join(
    broadcast(df_products),
    on=["category", "product_name"],
    how="inner"
)

# Xem execution plan để xác nhận broadcast join được sử dụng
df_broadcast_join.explain()

**Giải thích Broadcast Join:**

- `from pyspark.sql.functions import broadcast`: Import hàm `broadcast`.
- `broadcast(df_products)`: Gợi ý (hint) cho Spark Catalyst Optimizer rằng bảng `df_products` đủ nhỏ để có thể gửi (broadcast) toàn bộ dữ liệu của nó tới tất cả các worker node trong cluster, thay vì phải shuffle dữ liệu của cả 2 bảng lớn qua mạng.
- `on=["category", "product_name"]`: Khi cần join bằng nhiều cột, ta truyền vào một danh sách (list) các tên cột thay vì một chuỗi duy nhất.
- `.explain()`: In ra **Physical Plan** (kế hoạch thực thi vật lý) của Spark. Đây không phải là action chạy tính toán, mà chỉ hiển thị cách Spark dự định xử lý dữ liệu.

**Đầu ra mong đợi:** Console sẽ in ra một cây kế hoạch thực thi (dạng text). Bạn cần tìm từ khóa `BroadcastHashJoin` hoặc `BroadcastExchange` trong output đó. Nếu thấy, nghĩa là Spark đã nghe theo hint và sử dụng chiến lược Broadcast Join, giúp tiết kiệm rất nhiều thời gian và tài nguyên mạng.

In [ ]:
# Self join: Tìm các cặp khách hàng đã mua cùng một sản phẩm
s1 = df_sales.alias("s1")
s2 = df_sales.alias("s2")

df_self_join = s1.join(
    s2,
    (s1["product_name"] == s2["product_name"]) & 
    (s1["customer_id"] != s2["customer_id"]),
    how="inner"
).select(
    s1["customer_id"].alias("customer_1"),
    s2["customer_id"].alias("customer_2"),
    s1["product_name"]
).distinct()

df_self_join.show(10)

**Giải thích Self Join:**

- `.alias("s1")` và `.alias("s2")`: Vì ta join bảng `df_sales` với chính nó, Spark sẽ bị nhầm lẫn tên cột. `alias` giúp tạo ra 2 "bí danh" phân biệt cho cùng một DataFrame gốc.
- Điều kiện join: `(s1["product_name"] == s2["product_name"]) & (s1["customer_id"] != s2["customer_id"])`:
  - Phần 1: Sản phẩm mua phải giống nhau.
  - Phần 2: Khách hàng phải khác nhau (để tránh việc một khách hàng tự join với chính mình).
  - Lưu ý: Trong PySpark, điều kiện AND phải dùng dấu `&` chứ không dùng từ khóa `and` của Python, và mỗi vế phải bọc trong ngoặc đơn `()`.
- `.select(...)`: Chỉ chọn lấy 3 cột cần thiết để hiển thị, đồng thời đổi tên `customer_id` của s1 thành `customer_1`, s2 thành `customer_2` cho rõ nghĩa.
- `.distinct()`: Loại bỏ các cặp trùng lặp. Ví dụ nếu A và B cùng mua X, có thể sinh ra 2 dòng (A,B) và (B,A). Distinct giúp làm sạch kết quả.
- `.show(10)`: Hiển thị tối đa 10 dòng đầu tiên.

**Đầu ra mong đợi:** Bảng gồm 3 cột: `customer_1`, `customer_2`, `product_name`. Mỗi dòng đại diện cho một cặp khách hàng khác nhau đã từng mua cùng một sản phẩm. Tối đa 10 dòng được in ra màn hình.

## Tổng kết

Dưới đây là bảng tóm tắt các kiến thức và kỹ thuật đã thực hành trong bài tập này:

| Task | Nội dung | Kỹ thuật / Hàm chính |
|------|----------|----------------------|
| 3.1 | Khởi tạo môi trường và tải dữ liệu | `SparkSession.builder`, `read.csv`, `createOrReplaceTempView` |
| 3.1 | Truy vấn doanh số theo hạng thành viên | `JOIN`, `GROUP BY`, `COUNT`, `SUM`, `AVG`, `ORDER BY` |
| 3.2 | Phân tích CLV theo nhóm tuổi | `CASE WHEN`, `COUNT(DISTINCT)`, `GROUP BY` nhiều cột |
| 3.2 | Hiệu suất sản phẩm và biên lợi nhuận | `JOIN` nhiều điều kiện, `HAVING`, tính toán cột phái sinh |
| 3.2 | Phân tích xu hướng theo mùa | CTE (`WITH`), `EXTRACT`, Window Function `LAG()`, `PARTITION BY` |
| 3.3 | Thực hành Inner Join | `df.join(how="inner")`, `.count()` |
| 3.3 | Thực hành Left Join | `df.join(how="left")`, `.count()` |
| 3.3 | Thực hành Broadcast Join | `broadcast()`, join nhiều cột, `.explain()` |
| 3.3 | Thực hành Self Join | `.alias()`, điều kiện `&`, `.distinct()`, `.show(n)` |

Qua bài tập này, bạn đã nắm được cách chuyển đổi qua lại giữa tư duy SQL thuần túy và PySpark DataFrame API, cũng như hiểu được bản chất của các loại Join trong xử lý dữ liệu phân tán.